<a href="https://colab.research.google.com/github/sandhya14-automation/RoBERTa_News_Topic_Classifier_Transformer_Project/blob/main/Transformer_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**RoBERTa News Topic Classifier (AG News)**

In [ ]:
# 1. Install Required Libraries

!pip install --upgrade transformers datasets evaluate optuna accelerate -q

In [ ]:
# 2. Import Libraries

from datasets import load_dataset
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
import optuna
import torch

In [ ]:
 # 3. Load the AG News Dataset

dataset = load_dataset("ag_news")

In [ ]:
# 4. Load Tokenizer

model_name = "distilroberta-base"
tokenizer = RobertaTokenizerFast.from_pretrained(model_name)

In [ ]:
# 5. Tokenize the Dataset

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# 6. Use a Smaller Training Subset (20,000 samples)

small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(20000))

In [ ]:
# 7. Define Evaluation Metric (Accuracy + Macro‑F1)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [ ]:
# 8. Optuna Hyperparameter Tuning

def objective(trial):

    # 1. Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-6, 5e-5)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    num_epochs = trial.suggest_int("num_epochs", 1, 3)

    # 2. Training arguments using trial values
    training_args = TrainingArguments(
        output_dir="optuna-roberta",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none"
    )

    # 3. Create model + trainer
    model = RobertaForSequenceClassification.from_pretrained(
        model_name, num_labels=4, ignore_mismatched_sizes=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=small_train,
        eval_dataset=tokenized_dataset["test"],
        compute_metrics=compute_metrics
    )

    # 4. Train + evaluate
    trainer.train()
    eval_results = trainer.evaluate()

    # 5. Return macro-F1 to maximize
    return eval_results["eval_macro_f1"]


# Run Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=3)

print("Best hyperparameters:", study.best_params)

In [ ]:
# 9. Train Final Model Using Best Hyperparameters

best = study.best_params

training_args = TrainingArguments(
    output_dir="roberta-news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best["learning_rate"],
    per_device_train_batch_size=best["batch_size"],
    per_device_eval_batch_size=best["batch_size"],
    num_train_epochs=best["num_epochs"],
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True
)

model = RobertaForSequenceClassification.from_pretrained(
    model_name, num_labels=4, ignore_mismatched_sizes=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# 10. Evaluate the Model

trainer.evaluate()

In [ ]:
# 11. Predict Headline function

def predict_headline(text):
    # tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # move inputs to same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # forward pass
    outputs = model(**inputs)

    # prediction
    pred = torch.argmax(outputs.logits, dim=1).item()

    labels = ["World", "Sports", "Business", "Sci/Tech"]
    return labels[pred]

predict_headline("Apple releases new iPhone model")

In [ ]:
# 12. Confusion Matrix

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

preds = trainer.predict(tokenized_dataset["test"])
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["World", "Sports", "Business", "Sci/Tech"],
            yticklabels=["World", "Sports", "Business", "Sci/Tech"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# 13. Save Model for Later Use

model.save_pretrained("final_roberta_model")
tokenizer.save_pretrained("final_roberta_model")